# 📊 Exploratory Data Analysis & Advanced Feature Engineering
**Project: Trending Content Classifier**

This notebook performs the Exploratory Data Analysis (EDA) on the combined YouTube and TMDB dataset, verifies the target distributions, and analyzes the feature correlations and target encodings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

df = pd.read_csv('../data/youtube_trending.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 1. Class Balance Check
Checking the target distribution (`is_trending`) to verify balance.

In [ ]:
class_counts = df['is_trending'].value_counts()
class_pct = df['is_trending'].value_counts(normalize=True) * 100
for c in class_counts.index:
    print(f"Class {c}: {class_counts[c]} records ({class_pct[c]:.2f}%)")

## 2. Text clickbait & Title length analysis
We investigate if clickbait parameters (title length, caps ratio) correlate with trending status.

In [ ]:
df['title_length'] = df['title'].fillna('').apply(len)
df['title_caps_ratio'] = df['title'].fillna('').apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))

print("Mean Title Length for Trending:", df[df['is_trending'] == 1]['title_length'].mean())
print("Mean Title Length for Non-Trending:", df[df['is_trending'] == 0]['title_length'].mean())

print("Mean Caps Ratio for Trending:", df[df['is_trending'] == 1]['title_caps_ratio'].mean())
print("Mean Caps Ratio for Non-Trending:", df[df['is_trending'] == 0]['title_caps_ratio'].mean())

## 3. Genre target encoding visualization
Smoothing target encodings prevents overfitting on low-frequency categories.

In [ ]:
global_mean = df['is_trending'].mean()
genre_stats = df.groupby('genre')['is_trending'].agg(['count', 'mean'])
# Smoothed formula: (sum + prior * global) / (count + prior)
genre_stats['smoothed'] = (genre_stats['mean'] * genre_stats['count'] + 10 * global_mean) / (genre_stats['count'] + 10)
genre_stats.sort_values(by='smoothed', ascending=False)

## 4. Feature correlation matrix
Analyzing correlations of numerical and cyclical features to ensure we don't introduce redundant multicollinearity.

In [ ]:
numeric_cols = ['duration_minutes', 'title_length', 'title_caps_ratio', 'is_trending']
corr = df[numeric_cols].corr()
corr